In [ ]:
!git clone https://github.com/Ga0512/DeepMeca-Fire.git

Cloning into 'DeepMeca-Fire'...
remote: Enumerating objects: 16, done.
remote: Counting objects: 100% (16/16), done.
remote: Compressing objects: 100% (16/16), done.
remote: Total 16 (delta 4), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (16/16), 6.09 MiB | 7.19 MiB/s, done.
Resolving deltas: 100% (4/4), done.


In [ ]:
%cd DeepMeca-Fire

/content/DeepMeca-Fire


In [ ]:
!pip uninstall numpy scikit-learn -y
!pip install numpy==1.24.4 scikit-learn==1.2.2

Found existing installation: numpy 2.0.2
Uninstalling numpy-2.0.2:
  Successfully uninstalled numpy-2.0.2
Found existing installation: scikit-learn 1.6.1
Uninstalling scikit-learn-1.6.1:
  Successfully uninstalled scikit-learn-1.6.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.3/17.3 MB 82.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.6/9.6 MB 107.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pymc 5.23.0 requires numpy>=1.25.0, but you have numpy 1.24.4 which is incompatible.
mlxtend 0.23.4 requires scikit-learn>=1.3.1, but you have scikit-learn 1.2.2 which is incompatible.
tensorflow 2.18.0 requires numpy<2.1.0,>=1.26.0, but you have numpy 1.24.4 which is incompatible.
xarray-einstats 0.9.0 requires numpy>=1.25, but you have numpy 1.24.4 which is incompatible.
jaxlib 0.5.1 requires numpy>=1.25, but you have numpy 1.24.

In [ ]:
!pip install ultralytics fastapi uvicorn python-multipart opencv-python-headless matplotlib ultralytics pyngrok nest-asyncio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 48.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 38.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 83.9 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling 

In [ ]:
import cv2
import numpy as np
import requests
from fastapi import FastAPI, File, UploadFile
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse
from pydantic import BaseModel
import pickle
import bz2
from sklearn.preprocessing import StandardScaler
import pandas as pd
import time
import mimetypes

# Inicializar FastAPI
app = FastAPI()

# Middleware CORS
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# Carregar o modelo YOLO
from ultralytics import YOLO
model = YOLO("fire_n.pt")

# Carregar o modelo de regressão para previsão do FWI
model_R = pickle.load(bz2.BZ2File('regression.pkl', 'rb'))  # Carregar o modelo de regressão
scaler = StandardScaler()

df = pd.read_csv('Algerian_forest_fires_dataset_CLEANED.csv')
features = ['Temperature', 'Ws', 'FFMC', 'DMC', 'ISI']
df = df[features]

ACCESS_TOKEN = "XXXXX"
PHONE_NUMBER_ID = "XXX"

scaler.fit(df)

# Definindo um modelo Pydantic para garantir que os dados venham no formato correto
class PredictRequest(BaseModel):
    Temperature: float
    Ws: float
    FFMC: float
    DMC: float
    ISI: float
    whatsapp_number: str  # Número de WhatsApp do usuário

# Função para enviar mensagens de texto via WhatsApp
def send_whatsapp_message(to: str, message: str):
    url = f"https://graph.facebook.com/v22.0/{PHONE_NUMBER_ID}/messages"
    headers = {
        'Authorization': f'Bearer {ACCESS_TOKEN}',
        'Content-Type': 'application/json',
    }
    payload = {
        "messaging_product": "whatsapp",
        "to": to,
        "text": {"body": message},
    }
    response = requests.post(url, headers=headers, json=payload)
    if response.status_code == 200:
        print("Mensagem enviada com sucesso!")
        time.sleep(1)
    else:
        print(f"Erro ao enviar mensagem: {response.text}")


def upload_media(image_path: str) -> str:
    """Faz upload da imagem e retorna o media_id"""
    mime_type = mimetypes.guess_type(image_path)[0] or "application/octet-stream"
    file_name = image_path.split('/')[-1]

    upload_url = f"https://graph.facebook.com/{API_VERSION}/{PHONE_NUMBER_ID}/media"

    with open(image_path, 'rb') as file:
        files = {'file': (file_name, file, mime_type)}
        data = {'messaging_product': 'whatsapp', 'type': mime_type}
        headers = {'Authorization': f'Bearer {ACCESS_TOKEN}'}

        response = requests.post(upload_url, headers=headers, files=files, data=data)
        response.raise_for_status()

    return response.json()['id']

def send_whatsapp_image(to: str, image_path: str) -> dict:
    """Envia imagem para número WhatsApp especificado"""
    media_id = upload_media(image_path)

    message_url = f"https://graph.facebook.com/{API_VERSION}/{PHONE_NUMBER_ID}/messages"
    payload = {
        "messaging_product": "whatsapp",
        "recipient_type": "individual",
        "to": to,
        "type": "image",
        "image": {"id": media_id, "caption": "Alerta de incêndio detectado!"}
    }
    headers = {
        'Authorization': f'Bearer {ACCESS_TOKEN}',
        'Content-Type': 'application/json'
    }

    response = requests.post(message_url, json=payload, headers=headers)
    response.raise_for_status()

    return response.json()


# Endpoint para previsão de FWI
@app.post("/predict/")
async def predict_fwi(request: PredictRequest):
    # Preparar os dados de entrada para o modelo de regressão
    mock_data = np.array([[request.Temperature, request.Ws, request.FFMC, request.DMC, request.ISI]])

    # Escalar os dados
    scaled_data = scaler.transform(mock_data)

    # Prever o FWI
    regression_result = model_R.predict(scaled_data)[0]

    # Criar a mensagem com a previsão
    message = f"A previsão do FWI é: {round(regression_result, 2)}"


    send_whatsapp_message(str(request.whatsapp_number), message)

    # Retornar a previsão
    return JSONResponse(content={"FWI_estimation": round(regression_result, 2)})

# Endpoint para detecção de imagem
@app.post("/detect/")
async def detect_image(file: UploadFile = File(...)):
    # Ler a imagem do upload
    contents = await file.read()
    nparr = np.frombuffer(contents, np.uint8)
    img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)

    # Inferência do modelo YOLO
    results = model(img)[0]

    # Obter as caixas, classes e confiabilidade
    detections = []
    boxes = results.boxes

    for box in boxes:
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        cls_id = int(box.cls[0])
        conf = float(box.conf[0])

        # Desenhar a caixa delimitadora na imagem
        color = (0, 255, 0)  # Verde para as caixas
        thickness = 2  # Espessura da caixa
        img = cv2.rectangle(img, (int(x1), int(y1)), (int(x2), int(y2)), color, thickness)

        # Adicionar as informações de detecção
        detections.append({
            "class": model.names[cls_id],
            "conf": round(conf, 4),
            "box": [round(x1, 2), round(y1, 2), round(x2, 2), round(y2, 2)]
        })

    # Salvar a imagem com as caixas desenhadas
    output_path = "/content/detected_image_with_boxes.jpg"
    cv2.imwrite(output_path, img)

    # Enviar a imagem via WhatsApp
    if len(detections) > 0:
        send_whatsapp_image("5519996194036", output_path)

    return JSONResponse(content={"detections": detections})

# Endpoint de saúde
@app.get("/health")
def health_checker():
    return {"status": "healthy"}


/usr/local/lib/python3.11/dist-packages/sklearn/base.py:318: UserWarning: Trying to unpickle estimator DecisionTreeRegressor from version 1.0.2 when using version 1.2.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/base.py:318: UserWarning: Trying to unpickle estimator RandomForestRegressor from version 1.0.2 when using version 1.2.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [ ]:
from fastapi import FastAPI, File, UploadFile
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse
from ultralytics import YOLO
import cv2
import numpy as np
from pyngrok import ngrok
import nest_asyncio
import uvicorn
from pydantic import BaseModel

In [ ]:
# NGROK setup
ngrok.set_auth_token("2x3852dNlAvSpZDze9flupnAise_51iVVD1qPAmnjeya9tU6y")
public_url = ngrok.connect(9192).public_url
print(f"🚀 Running at: {public_url}")

# Para rodar em ambientes interativos (ex: Colab, notebooks)
nest_asyncio.apply()
uvicorn.run(app, host='0.0.0.0', port=9192)

🚀 Running at: https://e2b3-34-172-57-150.ngrok-free.app


INFO:     Started server process [506]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:9192 (Press CTRL+C to quit)


INFO:     179.191.248.113:0 - "GET / HTTP/1.1" 404 Not Found


ERROR:asyncio:Task exception was never retrieved
future: <Task finished name='Task-86' coro=<Server.serve() done, defined at /usr/local/lib/python3.11/dist-packages/uvicorn/server.py:68> exception=KeyboardInterrupt()>
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/uvicorn/main.py", line 580, in run
    server.run()
  File "/usr/local/lib/python3.11/dist-packages/uvicorn/server.py", line 66, in run
    return asyncio.run(self.serve(sockets=sockets))
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/nest_asyncio.py", line 30, in run
    return loop.run_until_complete(task)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/nest_asyncio.py", line 92, in run_until_complete
    self._run_once()
  File "/usr/local/lib/python3.11/dist-packages/nest_asyncio.py", line 133, in _run_once
    handle._run()
  File "/usr/lib/python3.11/asyncio/events.py", line 84, in _run
    s


0: 416x640 1 smoke, 1 fire, 231.9ms
Speed: 5.8ms preprocess, 231.9ms inference, 1.7ms postprocess per image at shape (1, 3, 416, 640)
INFO:     2804:431:9719:7244:95ee:b90d:c197:6c8c:0 - "POST /detect/ HTTP/1.1" 200 OK
INFO:     2804:431:9719:7244:95ee:b90d:c197:6c8c:0 - "POST /detect HTTP/1.1" 307 Temporary Redirect

0: 416x640 1 smoke, 1 fire, 168.9ms
Speed: 7.2ms preprocess, 168.9ms inference, 1.2ms postprocess per image at shape (1, 3, 416, 640)
INFO:     2804:431:9719:7244:95ee:b90d:c197:6c8c:0 - "POST /detect/ HTTP/1.1" 200 OK
INFO:     177.95.198.44:0 - "POST /detect HTTP/1.1" 307 Temporary Redirect

0: 416x640 1 smoke, 1 fire, 170.8ms
Speed: 5.8ms preprocess, 170.8ms inference, 1.3ms postprocess per image at shape (1, 3, 416, 640)
INFO:     2804:431:9719:86af:95ee:b90d:c197:6c8c:0 - "POST /detect/ HTTP/1.1" 500 Internal Server Error


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/uvicorn/protocols/http/h11_impl.py", line 403, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/uvicorn/middleware/proxy_headers.py", line 60, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/fastapi/applications.py", line 1054, in __call__
    await super().__call__(scope, receive, send)
  File "/usr/local/lib/python3.11/dist-packages/starlette/applications.py", line 112, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/usr/local/lib/python3.11/dist-packages/starlette/middleware/errors.py", line 187, in __call__
    raise exc
  File "/usr/local/lib/python3.11/dist-packages/starlette/middleware/errors.py",